# 01 · Análisis exploratorio de la temporada de Boca

Resultados, goles, localía y xG de la temporada, leídos de la API pública. Las conclusiones se imprimen con los números del momento: si cambian los datos, cambian las frases.

In [1]:
import sys, statistics, math
sys.path.insert(0, '.')
from datos import api, league_games, result, log_loss, elo_probabilities, m, RESULTS
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
partidos = api('boca_match', order='match_date')
res = ['G' if p['goals_for'] > p['goals_against'] else 'E' if p['goals_for'] == p['goals_against'] else 'P' for p in partidos]
pts = [3 if r == 'G' else 1 if r == 'E' else 0 for r in res]
print(f"{len(partidos)} partidos: {res.count('G')} G, {res.count('E')} E, {res.count('P')} P · {sum(pts)/len(pts):.2f} puntos por partido")
print(f"Goles por partido: {statistics.mean(p['goals_for'] for p in partidos):.2f} a favor, {statistics.mean(p['goals_against'] for p in partidos):.2f} en contra")

48 partidos: 24 G, 15 E, 9 P · 1.81 puntos por partido
Goles por partido: 1.46 a favor, 0.79 en contra


In [3]:
por_comp = {}
for p, r in zip(partidos, res):
    por_comp.setdefault(p['competition_name'], []).append(r)
for comp, rs in sorted(por_comp.items(), key=lambda x: -len(x[1])):
    print(f"{comp:40s} {len(rs):3d} PJ · {rs.count('G')} G {rs.count('E')} E {rs.count('P')} P")

Liga Profesional: Torneo Apertura         17 PJ · 8 G 6 E 3 P
Liga Profesional: Torneo Clausura         14 PJ · 7 G 6 E 1 P
CONMEBOL Copa Sudamericana                 8 PJ · 5 G 1 E 2 P
CONMEBOL Copa Libertadores                 6 PJ · 2 G 1 E 3 P
Copa Argentina                             3 PJ · 2 G 1 E 0 P


In [4]:
for local in (True, False):
    sub = [pt for p, pt in zip(partidos, pts) if p['boca_home'] == local]
    print(f"{'Local' if local else 'Visitante':10s} {len(sub)} PJ · {statistics.mean(sub):.2f} puntos por partido")

Local      22 PJ · 1.95 puntos por partido
Visitante  26 PJ · 1.69 puntos por partido


In [5]:
equipo = api('team_match_master', 'match_id,goals_for,goals_against,stats', order='match_date')
pares = [(e['stats'].get('xg'), e['goals_for']) for e in equipo if isinstance((e['stats'] or {}).get('xg'), (int, float))]
xg, gol = zip(*pares)
r = statistics.correlation(xg, gol)
print(f"{len(pares)} partidos con xG · xG medio {statistics.mean(xg):.2f}, goles medios {statistics.mean(gol):.2f}, correlación {r:.2f}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(xg, gol, color='#328336')
ax.plot([0, max(xg)], [0, max(xg)], color='#9AA09A', linestyle='--')
ax.set_xlabel('xG de Boca'); ax.set_ylabel('Goles de Boca'); ax.set_title('Goles contra xG por partido')
plt.show()

37 partidos con xG · xG medio 1.50, goles medios 1.41, correlación 0.40


C:\Users\Pato\AppData\Local\Temp\ipykernel_26296\1361565655.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura.** La correlación entre xG y goles por partido mide cuánto del resultado se explica por la calidad de las ocasiones; la diferencia acumulada (goles − xG) es la que el sitio muestra como «Rendimiento vs esperado».